In [ ]:
# Install the pinned runtime used by this repository
!pip -q install -r requirements.txt


## API key setup

Use Colab Secrets when possible. Add a secret named `OPENAI_API_KEY`. If unavailable, the cell will prompt securely.

In [ ]:
import os

try:
    from google.colab import userdata
    key = userdata.get('OPENAI_API_KEY')
    if key:
        os.environ['OPENAI_API_KEY'] = key
except Exception:
    pass

if not os.getenv('OPENAI_API_KEY'):
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OPENAI_API_KEY: ')

print('OPENAI_API_KEY is set:', bool(os.getenv('OPENAI_API_KEY')))

In [ ]:
# Optional: mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except Exception as e:
    print('Drive mount skipped:', e)

## Load the evaluation implementation

The full implementation is versioned in `trace_eval.py` so the notebook and local Python runs use the same code.


In [ ]:
from pathlib import Path

module_path = Path('trace_eval.py')
if not module_path.exists():
    raise FileNotFoundError('Run this notebook from the TRACE-Eval repository root.')
print(f'Using evaluation module: {module_path.resolve()}')


In [ ]:
from trace_eval import (
    EvalConfig,
    evaluate_extraction_pair_item_level,
    load_review_dashboard,
    launch_expert_review_dashboard,
    compute_final_metrics_from_review,
    save_final_adjudicated_metrics,
    save_adjudicated_metrics,
)

print('evaluation framework loaded.')


## Configure paths

Change these paths to your GT and generated extraction files.

In [ ]:
study_id = "tanner"

generated_docx_path = "examples/extracted/tanner_extraction_4o.docx"
groundtruth_docx_path = "examples/reference/tanner.docx"


output_dir = "outputs/tanner"

config = EvalConfig(
    extraction_model="gpt-5.1",
    judge_model="gpt-5.1",
    embedding_provider="openai",
    openai_embedding_model="text-embedding-3-large",
    top_k=5,
    min_candidate_similarity=0.75,
    sparse_min_candidate_similarity=0.50,
    lexical_fallback_top_k=5,
    lexical_min_score=0.50,
)

print('Configured.')

## Run item-level evaluation

This may take time because the extraction and judging are LLM calls.

In [ ]:
result = evaluate_extraction_pair_item_level(
    groundtruth_docx_path=groundtruth_docx_path,
    generated_docx_path=generated_docx_path,
    output_dir=output_dir,
    study_id=study_id,
    config=config,
)

print('Evaluation complete.')
print(result['metrics'])
print(result['output_paths'])

## Inspect dashboard outputs

In [ ]:
import pandas as pd
from IPython.display import display

paths = result['output_paths']
review_df = pd.read_csv(paths['review_csv']).fillna('')
summary_df = pd.read_excel(paths['excel'], sheet_name='summary_metrics')

print('Summary metrics')
display(summary_df)

print('Human review dashboard')
display(review_df.head(20))

## Launch expert review dashboard


In [ ]:
review_save_path = f"{output_dir}/{study_id}_adjudicated_review_in_progress.csv"
editable_review_df = launch_expert_review_dashboard(review_df, save_csv_path=review_save_path)

## Calculate final metrics after expert review

Run this after saving from the widget or manually editing the dashboard CSV.

The final output contract is restored to:

```python
{
    'csv':   '.../{study_id}_final_adjudicated_review.csv',
    'excel': '.../{study_id}_final_adjudicated_metrics.xlsx',
    'json':  '.../{study_id}_final_adjudicated_metrics.json',
}
```


In [ ]:
import pandas as pd
from IPython.display import display

# Load the expert-edited dashboard. If you did not edit every row, blank decisions
# are allowed: the system falls back to the LLM/system provisional label.
adjudicated_df = pd.read_csv(review_save_path).fillna('')

final_metrics = compute_final_metrics_from_review(adjudicated_df)
print('Final metrics after expert adjudication / fallback to LLM labels:')
print(final_metrics)

final_paths = save_final_adjudicated_metrics(
    review_df=adjudicated_df,
    output_dir=output_dir,
    study_id=study_id,
)

print('Final adjudicated output paths:')
print(final_paths)

summary_df = pd.read_excel(final_paths['excel'], sheet_name='summary_metrics')
display(summary_df)


## Download outputs in Colab

In [ ]:
try:
    from google.colab import files
    paths_to_download = [
        result['output_paths']['excel'],
        result['output_paths']['review_csv'],
        result['output_paths']['html'],
    ]
    if 'final_paths' in globals():
        paths_to_download.extend([final_paths['csv'], final_paths['excel'], final_paths['json']])
    for p in paths_to_download:
        files.download(p)
except Exception as e:
    print('Download skipped:', e)
